In [13]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gs
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [14]:
data = pd.read_excel('/content/superstore_dataset.xlsx')
data['Order Date'] = pd.to_datetime(data['Order Date'])
data['Ship Date'] = pd.to_datetime(data['Ship Date'])

In [15]:
data.head(2)

,Order ID,Order Date,Ship Date,Ship Mode,Segment,Region,State,Category,Sub-Category,Product ID,Product Name,Sales,Quantity,Discount,Profit,Unit Price,Lead Time (Days)
0,CA-2021-15678,2021-01-01,2021-01-04,Same Day,Home Office,East,New York,Technology,Accessories,TEC-AC-001,Logitech Keyboard,1303.90,13,0.0,127.43,100.30,1
1,CA-2021-14512,2021-01-01,2021-01-04,Second Class,Corporate,Central,Texas,Technology,Phones,TEC-PH-001,Samsung Galaxy,2604.99,3,0.1,96.35,964.81,3


In [16]:
data.shape

(9000, 17)

In [17]:
list(data.columns)

['Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Segment',
 'Region',
 'State',
 'Category',
 'Sub-Category',
 'Product ID',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit',
 'Unit Price',
 'Lead Time (Days)']

In [18]:
data.describe()

,Order Date,Ship Date,Sales,Quantity,Discount,Profit,Unit Price,Lead Time (Days)
count,9000,9000,9000.000000,9000.000000,9000.000000,9000.000000,9000.000000,9000.000000
mean,2022-06-30 23:48:38.400000,2022-07-04 17:05:07.200000,3128.803133,7.444667,0.099767,478.848732,472.009173,2.729556
min,2021-01-01 00:00:00,2021-01-02 00:00:00,13.410000,1.000000,0.000000,-1472.020000,18.710000,1.000000
25%,2021-09-24 00:00:00,2021-09-28 00:00:00,409.177500,4.000000,0.000000,23.080000,91.567500,1.000000
50%,2022-06-30 12:00:00,2022-07-05 00:00:00,1288.990000,7.000000,0.100000,125.530000,274.355000,2.000000
75%,2023-04-05 00:00:00,2023-04-08 06:00:00,3685.637500,11.000000,0.200000,481.595000,565.027500,3.000000
max,2023-12-31 00:00:00,2024-01-05 00:00:00,32442.760000,14.000000,0.300000,9814.530000,2339.700000,5.000000
std,NaN,NaN,4689.477768,4.030477,0.115366,994.142030,585.702689,1.482413


In [19]:
data.isnull().sum()

,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Segment,0
Region,0
State,0
Category,0
Sub-Category,0
Product ID,0


In [20]:
BG='#0F1923';CARD='#1A2635';ACCENT1='#00C2FF'; ACCENT2='#FF6B6B';
ACCENT3='#00E5A0';ACCENT4='#FFB347';TEXT='#E8EDF2'; MUTED='#8899AA';

from typing import Text
data['YearMonth'] = data['Order Date'].dt.to_period('M')

# Grouping data by product, region, and month to create the 'monthly' DataFrame
monthly = data.groupby(['Product ID','Product Name','Category','Sub-Category','Region','YearMonth']).\
agg(
    Total_Quantity = ('Quantity','sum'),
    Total_Sales = ('Sales','sum'),
    Avg_Discount = ('Discount','mean'),
    Avg_Profit = ('Profit','mean'),
    Avg_Unit_Price = ('Unit Price','mean'),
    Num_Orders = ('Order ID','count'),
    Avg_Lead_Time = ('Lead Time (Days)','mean')
).reset_index()

# Further processing of 'monthly' for 'Month' column and sorting
monthly = monthly.sort_values(['Product ID','YearMonth']).reset_index(drop=True)
monthly['YearMonth_dt'] = monthly['YearMonth'].dt.to_timestamp()
monthly['Month']= monthly['YearMonth_dt'].dt.month
monthly['Year']= monthly['YearMonth_dt'].dt.year
monthly['Quarter'] = monthly['YearMonth_dt'].dt.quarter

fig = plt.figure(figsize=(10, 10),)
fig.patch.set_facecolor(BG)
fig.suptitle('Superstore Sales - EDA',fontsize=22,color=TEXT)
gse=gs.GridSpec(4,3, figure=fig, hspace=0.45,wspace=0.35,left=0.06,right=0.97,top=0.94,bottom=0.04)
def style_ax(ax,title):
  ax.set_facecolor(CARD)
  ax.tick_params(colors=MUTED,labelsize=8)
  for spine in ax.spines.values():
    spine.set_edgecolor('#2A3A4A')
  ax.set_title(title,color=TEXT, fontsize=10,fontweight='bold', pad=8)
  ax.grid(axis='y',color='#2A3A4A',linewidth=0.5,alpha=0.7)


  # monthly Sales trend
ax1 = fig.add_subplot(gse[0, :2])
monthly_sales_trend = data.groupby('YearMonth')['Sales'].sum().reset_index()
ax1.fill_between(range(len(monthly_sales_trend)),monthly_sales_trend['Sales']/1000,linewidth = 2.5,color=ACCENT1,alpha=0.6)
labels = [str(m) for m in monthly_sales_trend['YearMonth']]
ax1.set_xticks(range(0,len(monthly_sales_trend),3))
ax1.set_xticklabels(labels[::3],rotation=30)
ax1.set_ylabel('Sales (K USD)',fontsize=9)
style_ax(ax1,'Monthly Sales Trend (2021-2023)')


#category pie
ax2 = fig.add_subplot(gse[0, 2])
ax2.set_facecolor(CARD)
cat_sales = data.groupby('Category')['Sales'].sum()
wedges, texts, autotexts = ax2.pie(cat_sales, labels=cat_sales.index,
autopct='%1.1f%%',
    colors=[ACCENT1, ACCENT3, ACCENT4], startangle=140, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=2))
for t in texts: t.set_color(TEXT); t.set_fontsize(9)
for at in autotexts: at.set_color(BG); at.set_fontsize(8); at.set_fontweight('bold')
ax2.set_title('Category Sales',color=TEXT, fontsize=10,fontweight='bold', pad=8)


#qty by sub-category
# 3. Qty by Sub-Category
ax3 = fig.add_subplot(gse[1, :])
subcat_qty = data.groupby('Sub-Category')['Quantity'].sum().sort_values(ascending=True)
colors_bar = [ACCENT1 if v > subcat_qty.median() else ACCENT2 for v in subcat_qty]
bars = ax3.barh(subcat_qty.index, subcat_qty.values, color=colors_bar,
edgecolor='none', height=0.7)
for bar, val in zip(bars, subcat_qty.values):
    ax3.text(val+20, bar.get_y()+bar.get_height()/2, f'{val:,}', va='center',
color=MUTED, fontsize=8)
ax3.set_xlabel('Total Quantity Sold', color=MUTED, fontsize=9)
style_ax(ax3, 'Total Quantity Sold by Sub-Category')
ax3.set_yticklabels(subcat_qty.index, color=TEXT, fontsize=9)
ax3.grid(axis='x', color='#2A3A4A', linewidth=0.5, alpha=0.7)
ax3.grid(axis='y', visible=False)


#Demand variablity
ax4 = fig.add_subplot(gse[2, :2])
mq = monthly.groupby(['Sub-Category','Month'])['Total_Quantity'].sum().reset_index()
var = mq.groupby('Sub-Category')['Total_Quantity'].std().sort_values(ascending=False)
bc = [ACCENT2 if v > var.median() else ACCENT3 for v in var]
ax4.bar(var.index, var.values, color=bc, edgecolor='none', width=0.7)
ax4.set_xticklabels(var.index, rotation=40, ha='right', fontsize=8, color=MUTED)
ax4.set_ylabel('Std Dev Monthly Qty', color=MUTED, fontsize=9)
ax4.axhline(var.median(), color=ACCENT4, linestyle='--', linewidth=1.5,
label=f'Median: {var.median():.1f}')
ax4.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)
style_ax(ax4, 'Demand Variability by Sub-Category (Key for Safety Stock!)')

#region sales
ax5 = fig.add_subplot(gse[2, 2])
rs = data.groupby('Region')['Sales'].sum().sort_values(ascending=False)
ax5.bar(rs.index, rs.values/1000, color=[ACCENT1, ACCENT3, ACCENT4, ACCENT2],
edgecolor='none', width=0.6)
for i, (r, v) in enumerate(rs.items()):
    ax5.text(i, v/1000+50, f'${v/1000:.0f}K', ha='center', color=TEXT, fontsize=8,
fontweight='bold')
ax5.set_ylabel('Sales (K USD)', color=MUTED, fontsize=9)
ax5.set_xticklabels(rs.index, color=TEXT, fontsize=9)
style_ax(ax5, 'Region Sales')


#profit vs sales
ax7 = fig.add_subplot(gse[3, 2])
ax7.set_facecolor(CARD)
s = data.sample(500, random_state=42)
cs = [ACCENT3 if p > 0 else ACCENT2 for p in s['Profit']]
ax7.scatter(s['Sales']/1000, s['Profit']/1000, c=cs, alpha=0.6, s=20,
edgecolors='none')
ax7.axhline(0, color=ACCENT4, linewidth=1, linestyle='--', alpha=0.7)
ax7.set_xlabel('Sales (K USD)', color=MUTED, fontsize=9)
ax7.set_ylabel('Profit (K USD)', color=MUTED, fontsize=9)
style_ax(ax7, 'Profit vs Sales')
for spine in ax7.spines.values():
    spine.set_color('#2A3A4A')
plt.savefig('eda_eport.png',dpi=150,bbox_inches="tight", facecolor=BG)

plt.show()

In [21]:
#grouping data by product region and month
data['YearMonth'] = data['Order Date'].dt.to_period('M')
monthly = data.groupby(['Product ID','Product Name','Category','Sub-Category','Region','YearMonth']).\
agg(
    Total_Quantity = ('Quantity','sum'),
    Total_Sales = ('Sales','sum'),
    Avg_Discount = ('Discount','mean'),
    Avg_Profit = ('Profit','mean'),
    Avg_Unit_Price = ('Unit Price','mean'),
    Num_Orders = ('Order ID','count'),
    Avg_Lead_Time = ('Lead Time (Days)','mean')
).reset_index()

In [22]:
#monthly aggregation
monthly = monthly.sort_values(['Product ID','YearMonth']).reset_index(drop=True)
monthly['YearMonth_dt'] = monthly['YearMonth'].dt.to_timestamp()
monthly['Month']= monthly['YearMonth_dt'].dt.month
monthly['Year']= monthly['YearMonth_dt'].dt.year
monthly['Quarter'] = monthly['YearMonth_dt'].dt.quarter

print(f"Monthly records: {len(monthly)}")
monthly.head(3)

Monthly records: 1437


,Product ID,Product Name,Category,Sub-Category,Region,YearMonth,Total_Quantity,Total_Sales,Avg_Discount,Avg_Profit,Avg_Unit_Price,Num_Orders,Avg_Lead_Time,YearMonth_dt,Month,Year,Quarter
0,FUR-BO-001,Bush Bookcase,Furniture,Bookcases,Central,2021-01,50,14870.48,0.0600,244.652,300.6320,5,2.600,2021-01-01,1,2021,1
1,FUR-BO-001,Bush Bookcase,Furniture,Bookcases,East,2021-01,48,10455.22,0.1000,168.844,266.9280,5,1.800,2021-01-01,1,2021,1
2,FUR-BO-001,Bush Bookcase,Furniture,Bookcases,South,2021-01,60,13315.77,0.1875,155.850,286.5775,8,2.625,2021-01-01,1,2021,1


In [23]:

g = monthly.groupby('Product ID')['Total_Quantity']
for lag in [1,2,3,6,12]:
  monthly[f'lag_{lag}'] = g.shift(lag)
print('lags ok')
for w in [3,6,12]:
  monthly[f'RM_{w}'] = g.transform(lambda x: x.shift(1).rolling(w,min_periods=1).mean())
  monthly[f'RS_{w}'] = g.transform(lambda x: x.shift(1).rolling(w,min_periods=2).std())
print('rolling ok')

monthly['Is_Q4'] = (monthly['Quarter'] == 4).astype(int)
monthly['Is_summer'] = (monthly['Month'].isin([6,7,8])).astype(int)
monthly['Month_Index'] = (monthly['Year']-2021)*12 + monthly['Month']

std6 = monthly['RS_6'].fillna(monthly['RS_3']).fillna(0)
monthly['safety_Stock_95'] = (1.645*std6*np.sqrt(monthly['Avg_Lead_Time'])).round(1)
monthly['safety_Stock_95'] = (2.326*std6*np.sqrt(monthly['Avg_Lead_Time'])).round(1)

monthly['Reorder_Point'] = (monthly['RM_6']*monthly['Avg_Unit_Price']/30 + monthly['safety_Stock_95']).round(1)
monthly['Avg_Daily_Demand'] = (monthly['RM_6']/30).round(3)


monthly['Cat_Code'] = pd.Categorical(monthly['Category']).codes
monthly['SubCat_Code'] = pd.Categorical(monthly['Sub-Category']).codes
monthly = monthly.dropna(subset = ['lag_1']).reset_index(drop=True)
print("features ok, shape:", monthly.shape)


monthly.head(3)

lags ok
rolling ok
features ok, shape: (1427, 36)


,Product ID,Product Name,Category,Sub-Category,Region,YearMonth,Total_Quantity,Total_Sales,Avg_Discount,Avg_Profit,...,RM_12,RS_12,Is_Q4,Is_summer,Month_Index,safety_Stock_95,Reorder_Point,Avg_Daily_Demand,Cat_Code,SubCat_Code
0,FUR-BO-001,Bush Bookcase,Furniture,Bookcases,East,2021-01,48,10455.22,0.1000,168.84400,...,50.000000,NaN,0,0,1,0.0,444.9,1.667,0,3
1,FUR-BO-001,Bush Bookcase,Furniture,Bookcases,South,2021-01,60,13315.77,0.1875,155.85000,...,49.000000,1.414214,0,0,1,5.3,473.4,1.633,0,3
2,FUR-BO-001,Bush Bookcase,Furniture,Bookcases,West,2021-01,75,21490.21,0.0375,693.56875,...,52.666667,6.429101,0,0,1,25.4,540.5,1.756,0,3


In [24]:
# model and evalution
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [25]:
FEATURES = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'RM_3', 'RM_6', 'RM_12', 'RS_3', 'RS_6', 'RS_12',
    'Month', 'Year', 'Quarter', 'Is_Q4', 'Is_summer',
    'Month_Index', 'Cat_Code', 'SubCat_Code',
    'Avg_Lead_Time', 'Avg_Discount', 'Avg_Unit_Price', 'Num_Orders'
]
TARGET = 'Total_Quantity'

monthly[FEATURES] = monthly[FEATURES].fillna(0)

In [26]:
train = monthly[monthly['YearMonth'] < '2023-01'].copy()
test  = monthly[monthly['YearMonth'] >= '2023-01'].copy()
split_idx = len(train)

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

In [27]:
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=10))
])

gbr = GradientBoostingRegressor(
    n_estimators     = 300,
    max_depth        = 2,
    learning_rate    = 0.05,
    subsample        = 0.6,
    min_samples_leaf = 15,
    max_features     = 0.7,
    random_state     = 42
)

ridge.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()), ('model', Ridge(alpha=10))])

In [28]:
gbr.fit(X_train, y_train)

GradientBoostingRegressor(learning_rate=0.05, max_depth=2, max_features=0.7,
                          min_samples_leaf=15, n_estimators=300,
                          random_state=42, subsample=0.6)

In [29]:
def ensemble_predict(X):
    return np.maximum(
        0.55 * ridge.predict(X) + 0.45 * gbr.predict(X),
        0
    )

y_pred_train = ensemble_predict(X_train)
y_pred_test  = ensemble_predict(X_test)

In [30]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

train_r2  = r2_score(y_train, y_pred_train)
test_r2   = r2_score(y_test,  y_pred_test)
test_mae  = mean_absolute_error(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mape = mape(y_test.values, y_pred_test)
gap       = train_r2 - test_r2

In [31]:
[train_r2, test_r2, test_mae, test_rmse, test_mape, gap]

[0.8026619472849575,
 0.7878342020570867,
 7.861477743832046,
 np.float64(9.98118811123616),
 np.float64(21.61607644600716),
 0.014827745227870892]

In [32]:
tscv = TimeSeriesSplit(n_splits=3)

cv_ridge = cross_val_score(ridge, monthly[FEATURES], monthly[TARGET], cv=tscv, scoring='r2')
cv_gbr   = cross_val_score(gbr,   monthly[FEATURES], monthly[TARGET], cv=tscv, scoring='r2')

In [33]:
cv_ridge.mean(),cv_gbr.mean()

(np.float64(0.7458904709762173), np.float64(0.7250880688205683))

In [36]:
import joblib
joblib.dump(ridge, 'Sales forecasting.pkl')
from google.colab import files
files.download('Sales forecasting.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>